# NCAA March Madness Advanced Feature Engineering

This notebook implements advanced features:
1. **Strength of Schedule (SOS)**: Adjust stats based on opponent quality.
2. **Recent Trends**: Stats from the last 10 games of the season.

In [6]:
import pandas as pd
import numpy as np
import os

DATA_PATH = './march-machine-learning-mania-2026/'

def load_data(gender):
    prefix = gender.upper()
    teams = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}Teams.csv'))
    seeds = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}NCAATourneySeeds.csv'))
    regular_results = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}RegularSeasonDetailedResults.csv'))
    return teams, seeds, regular_results

m_teams, m_seeds, m_regular_results = load_data('M')
w_teams, w_seeds, w_regular_results = load_data('W')
m_rankings = pd.read_csv(os.path.join(DATA_PATH, 'MMasseyOrdinals.csv'))

print("Data loaded.")

Data loaded.


## 1. Helper Functions for SOS and Trends

In [7]:
def get_game_level_stats(df):
    # Calculate possessions per game
    df['WPoss'] = df['WFGA'] + 0.475 * df['WFTA'] - df['WOR'] + df['WTO']
    df['LPoss'] = df['LFGA'] + 0.475 * df['LFTA'] - df['LOR'] + df['LTO']
    
    # OffEff = (Points / Possessions) * 100
    df['WOffEff'] = (df['WScore'] / df['WPoss']) * 100
    df['LOffEff'] = (df['LScore'] / df['LPoss']) * 100
    
    # DefEff = Opponent's OffEff
    df['WDefEff'] = df['LOffEff']
    df['LDefEff'] = df['WOffEff']
    
    # Extract two rows per game
    w_data = df[['Season', 'DayNum', 'WTeamID', 'WOffEff', 'WDefEff', 'WScore']].rename(columns={'WTeamID': 'TeamID', 'WOffEff': 'OffEff', 'WDefEff': 'DefEff', 'WScore': 'Score'})
    l_data = df[['Season', 'DayNum', 'LTeamID', 'LOffEff', 'LDefEff', 'LScore']].rename(columns={'LTeamID': 'TeamID', 'LOffEff': 'OffEff', 'LDefEff': 'DefEff', 'LScore': 'Score'})
    
    # Add Opponent ID for SOS calculation later
    w_data['OppID'] = df['LTeamID']
    l_data['OppID'] = df['WTeamID']
    
    return pd.concat([w_data, l_data])

def calculate_advanced_stats(game_df):
    # Seasonal averages
    seasonal = game_df.groupby(['Season', 'TeamID']).agg({'OffEff': 'mean', 'DefEff': 'mean', 'Score': 'mean'}).reset_index()
    
    # SOS calculation: Average seasonal DefEff of all opponents played
    # First merge with seasonal stats to get opponent strength
    game_with_opp = game_df.merge(seasonal.rename(columns={'TeamID': 'OppID', 'OffEff': 'OppOffEff', 'DefEff': 'OppDefEff'}), on=['Season', 'OppID'])
    sos = game_with_opp.groupby(['Season', 'TeamID']).agg({'OppDefEff': 'mean', 'OppOffEff': 'mean'}).reset_index()
    sos.columns = ['Season', 'TeamID', 'SOS_Off', 'SOS_Def']
    
    # Recent Trend (Last 10 games)
    game_df = game_df.sort_values(['Season', 'TeamID', 'DayNum'])
    trend = game_df.groupby(['Season', 'TeamID']).tail(10).groupby(['Season', 'TeamID']).agg({'OffEff': 'mean', 'Score': 'mean'}).reset_index()
    trend.columns = ['Season', 'TeamID', 'TrendOffEff', 'TrendScore']
    
    final = seasonal.merge(sos, on=['Season', 'TeamID']).merge(trend, on=['Season', 'TeamID'])
    return final

print("Functions ready.")

Functions ready.


## 2. Processing Men's and Women's Features

In [8]:
def process_gender(regular_results, seeds, rankings=None):
    game_level = get_game_level_stats(regular_results)
    features = calculate_advanced_stats(game_level)
    
    seeds['SeedInt'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    features = features.merge(seeds[['Season', 'TeamID', 'SeedInt']], on=['Season', 'TeamID'], how='left')
    features['SeedInt'] = features['SeedInt'].fillna(20)
    
    if rankings is not None:
        m_final_rankings = rankings[rankings['RankingDayNum'] == 133].copy()
        m_consensus_rank = m_final_rankings.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index().rename(columns={'OrdinalRank': 'AvgRank'})
        features = features.merge(m_consensus_rank, on=['Season', 'TeamID'], how='left')
        features['AvgRank'] = features['AvgRank'].fillna(features['AvgRank'].max())
    else:
        features['AvgRank'] = features['SeedInt'] * 10
        
    return features

m_features = process_gender(m_regular_results, m_seeds, m_rankings)
w_features = process_gender(w_regular_results, w_seeds)

all_features = pd.concat([m_features, w_features], ignore_index=True)
all_features.to_csv('all_team_features_v2.csv', index=False)
print(f"v2 Features saved. Total rows: {len(all_features)}")

v2 Features saved. Total rows: 14311
